# Similarity-Guided Linear Programming for Control Action Selection

## Approach

This notebook implements a simple LP-based control action recommendation system:

1. **Similarity retrieval**: For a given plant state, find the K most similar historical situations from training data
2. **LP optimization**: Use Linear Programming to select the optimal action (tag + direction + magnitude) that:
   - Follows the consensus direction of similar historical actions (weighted by similarity score)
   - Respects move-size limits derived from historical data
   - Penalizes unnecessarily large moves
3. **Evaluation**: Compare LP-recommended actions against what operators actually did in test episodes

### Data Source
- `DATA/1071_pvlo_alarms_clustered_with_control_actions_with_plant_context.xlsx`
- Contains 7,246 unique merged OP/SP actions across 419 alarm clusters
- Each action has 107 plant-context features (`merged_ctx_*` columns)

### Train/Test Split
- **Training**: 2022–2024 episodes (used as the historical action library)
- **Testing**: 2025 episodes (evaluate recommendations against actual operator actions)

In [20]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

BASE_DIR = Path("/home/h604827/ControlActions")
WORKBOOK_PATH = BASE_DIR / "DATA/1071_pvlo_alarms_clustered_with_control_actions_with_plant_context.xlsx"

ALARM_THRESHOLD = 28.75
TOP_K = 10  # Number of similar historical actions to retrieve
MOVE_PENALTY = 0.1  # Lambda: penalizes large moves in the LP objective

EXCLUDED_TAGS = ["03FIC_3435"]  # Bypass path — client instruction to ignore
# Tags to exclude from recommendations (per domain knowledge)

## 1. Load and Prepare Data

In [21]:
# Load workbook sheets
alarm_clusters_df = pd.read_excel(WORKBOOK_PATH, sheet_name="alarm_clusters")
control_actions_df = pd.read_excel(WORKBOOK_PATH, sheet_name="control_actions")

# Filter to unique merged OP/SP actions only (one row per merged group)
actions_df = control_actions_df[
    control_actions_df["merged_group_role"].isin(["SINGLE", "START"])
    & control_actions_df["Description"].isin(["OP", "SP"])
].copy()

actions_df["cluster_start"] = pd.to_datetime(actions_df["cluster_start"])
actions_df["cluster_end"] = pd.to_datetime(actions_df["cluster_end"])
actions_df["merged_action_timestamp"] = pd.to_datetime(actions_df["merged_action_timestamp"])

# Remove excluded tags entirely from analysis
actions_df = actions_df[~actions_df["Source"].isin(EXCLUDED_TAGS)].reset_index(drop=True)

print(f"Total unique merged OP/SP actions: {len(actions_df):,}")
print(f"Unique clusters: {actions_df['cluster_id'].nunique()}")

Total unique merged OP/SP actions: 5,535
Unique clusters: 376


In [22]:
# Identify context feature columns (the plant state at action time)
ctx_columns = [c for c in actions_df.columns if c.startswith("merged_ctx_")]

# Use norm_pos columns as the primary similarity features (most complete coverage)
norm_pos_cols = [c for c in ctx_columns if "norm_pos" in c]
print(f"Context columns total: {len(ctx_columns)}")
print(f"Norm-position columns (for similarity): {len(norm_pos_cols)}")

# Filter to rows with complete norm_pos context
has_context = actions_df[norm_pos_cols].notna().all(axis=1)
actions_df = actions_df[has_context].copy().reset_index(drop=True)
print(f"Actions with complete context: {len(actions_df):,}")

Context columns total: 107
Norm-position columns (for similarity): 26
Actions with complete context: 5,101


In [23]:
# Train/test split: 2022-2024 for training, 2025 for testing
actions_df["year"] = actions_df["cluster_start"].dt.year

train_df = actions_df[actions_df["year"] <= 2024].copy().reset_index(drop=True)
test_df = actions_df[actions_df["year"] == 2025].copy().reset_index(drop=True)

print(f"Training actions: {len(train_df):,} ({train_df['cluster_id'].nunique()} clusters)")
print(f"Testing actions:  {len(test_df):,} ({test_df['cluster_id'].nunique()} clusters)")
print(f"\nTraining top sources:")
print(train_df["Source"].value_counts().head(8).to_string())
print(f"\nTest top sources:")
print(test_df["Source"].value_counts().head(8).to_string())

Training actions: 4,164 (298 clusters)
Testing actions:  937 (61 clusters)

Training top sources:
Source
03HIC_1151    711
03HIC_3100    541
03PIC_1013    459
03LIC_1034    391
03LIC_1071    375
03HIC_1141    335
03HIC_3132    235
03LIC_1016    233

Test top sources:
Source
03PIC_1013    181
03LIC_1071    154
03HIC_1151    116
03LIC_1034     91
03HIC_3100     78
03HIC_1141     51
03LIC_1085     51
03HIC_3132     43


## 2. Compute Move-Size Limits Per Tag

From training data, compute the typical range of step sizes for each tag.
These become constraints in the LP (we won't recommend moves larger than what operators historically did).

In [24]:
# Compute per-tag move limits from training data (P5/P95 of historical steps)
move_limits = (
    train_df.groupby("Source")["merged_step"]
    .agg(
        count="count",
        min_step=lambda x: x.quantile(0.05),
        max_step=lambda x: x.quantile(0.95),
        median_step="median",
        abs_median=lambda x: x.abs().median(),
    )
    .sort_values("count", ascending=False)
)

# Only consider tags with at least 10 training actions
MIN_ACTIONS = 10
move_limits = move_limits[move_limits["count"] >= MIN_ACTIONS].copy()

# The max absolute move we'll allow per tag
move_limits["max_abs_move"] = move_limits[["min_step", "max_step"]].abs().max(axis=1)

# Remove excluded tags
move_limits = move_limits.drop(index=[t for t in EXCLUDED_TAGS if t in move_limits.index], errors="ignore")


valid_tags = move_limits.index.tolist()
print(move_limits[["count", "min_step", "max_step", "abs_median", "max_abs_move"]].to_string())
print(f"Tags with sufficient training data: {len(valid_tags)}")

             count   min_step  max_step  abs_median  max_abs_move
Source                                                           
03HIC_1151     711  -5.500000   6.00000     2.00000      6.000000
03HIC_3100     541 -10.000000   8.00000     2.00000     10.000000
03PIC_1013     459  -4.000000   3.00000     2.00000      4.000000
03LIC_1034     391  -2.000000   1.75000     1.00000      2.000000
03LIC_1071     375 -21.755840  10.00000     3.00000     21.755840
03HIC_1141     335  -8.000000   9.30000     2.00000      9.300000
03HIC_3132     235  -4.000000  18.00000     2.00000     18.000000
03LIC_1016     233 -24.804180  15.00000     2.00000     24.804180
03FIC_3415     150 -22.200000  23.65000     8.00000     23.650000
03LIC_3153     122 -11.900000  37.60000     5.00000     37.600000
03PIC_3131     108 -48.802165   4.00000     4.00000     48.802165
03LIC_1085      85  -4.700040   1.90000     1.00000      4.700040
03FIC_1085      70  -5.692830  14.00000     3.34405     14.000000
03TIC_1009

## 3. Similarity Function

For a given test action's plant context, find the K most similar training actions using cosine similarity on the normalized position features.

In [25]:
# Precompute the training context matrix
# Filter training to only valid tags (ones we can recommend)
train_valid_df = train_df[train_df["Source"].isin(valid_tags)].copy().reset_index(drop=True)
train_context_matrix = train_valid_df[norm_pos_cols].values

print(f"Training library size (valid tags only): {len(train_valid_df):,}")
print(f"Context vector dimension: {train_context_matrix.shape[1]}")


def find_similar_actions(test_context_vector, top_k=TOP_K):
    """Find top-K most similar training actions by cosine similarity."""
    # Reshape for sklearn
    query = test_context_vector.reshape(1, -1)
    
    # Compute cosine similarity against all training contexts
    similarities = cosine_similarity(query, train_context_matrix)[0]
    
    # Get top-K indices
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    
    results = []
    for idx in top_k_indices:
        row = train_valid_df.iloc[idx]
        results.append({
            "source": row["Source"],
            "description": row["Description"],
            "step": row["merged_step"],
            "direction": row["merged_action_direction"],
            "similarity": similarities[idx],
            "cluster_id": row["cluster_id"],
            "alarm_proximity": row.get("merged_ctx_alarm_proximity", np.nan),
        })
    
    return pd.DataFrame(results)


# Quick test: find similar actions for the first test action
sample_context = test_df[norm_pos_cols].iloc[0].values
sample_neighbors = find_similar_actions(sample_context)
print("\nSample: top-10 similar actions for first test point:")
print(sample_neighbors.to_string(index=False))

Training library size (valid tags only): 4,137
Context vector dimension: 26

Sample: top-10 similar actions for first test point:
    source description   step direction  similarity  cluster_id  alarm_proximity
03HIC_3100          OP  1.000        up    0.893999         256         1.859110
03HIC_3100          OP  2.000        up    0.861719         456         1.249468
03FIC_1085          OP  0.200        up    0.857966         284         1.971959
03HIC_1141          OP  2.131        up    0.854838         269         1.402766
03LIC_1034          SP -1.000      down    0.840185         256         1.347439
03LIC_1034          SP -0.500      down    0.838979         262         2.169230
03HIC_1151          OP  5.000        up    0.826351         225         2.209314
03FIC_3415          OP 55.000        up    0.818552         205         1.989935
03FIC_1085          OP -0.600      down    0.817764         284         2.070568
03HIC_3100          OP  1.000        up    0.814356         

## 4. LP Formulation

Given top-K similar historical actions, the LP selects the best action:

**Decision variables:** For each candidate tag $j$, the step $\Delta u_j = \Delta u_j^+ - \Delta u_j^-$

**Objective (minimize):**
$$-\sum_j \text{score}_j \cdot d_j \cdot (\Delta u_j^+ - \Delta u_j^-) + \lambda \sum_j (\Delta u_j^+ + \Delta u_j^-)$$

- First term rewards following the similarity-weighted consensus direction
- Second term penalizes large moves (move suppression)

**Constraints:**
- $0 \leq \Delta u_j^+, \Delta u_j^- \leq \text{max\_move}_j$
- At most one of $\Delta u_j^+, \Delta u_j^-$ will be non-zero (ensured by the objective structure)

The LP selects the **single best tag** with the optimal step.

In [26]:
def solve_action_lp(similar_actions_df, move_penalty=MOVE_PENALTY):
    """
    Solve LP to select optimal control action given similar historical actions.
    
    Returns dict with: recommended_tag, recommended_direction, recommended_step, solver_status
    """
    # Aggregate recommendations per tag: similarity-weighted direction score
    tag_scores = {}
    for _, row in similar_actions_df.iterrows():
        tag = row["source"]
        sim = row["similarity"]
        step = row["step"]
        if pd.isna(step) or tag not in valid_tags:
            continue
        direction = 1.0 if step > 0 else (-1.0 if step < 0 else 0.0)
        
        if tag not in tag_scores:
            tag_scores[tag] = {"weighted_direction": 0.0, "total_sim": 0.0, "weighted_magnitude": 0.0, "count": 0}
        tag_scores[tag]["weighted_direction"] += sim * direction
        tag_scores[tag]["total_sim"] += sim
        tag_scores[tag]["weighted_magnitude"] += sim * abs(step)
        tag_scores[tag]["count"] += 1
    
    if not tag_scores:
        return {"recommended_tag": None, "recommended_step": 0.0, "recommended_direction": "none", "status": "no_candidates"}
    
    # Build LP: one pair of variables (u+, u-) per candidate tag
    tags = list(tag_scores.keys())
    n_tags = len(tags)
    
    # Variables: [u+_1, u-_1, u+_2, u-_2, ...] (2 * n_tags variables)
    n_vars = 2 * n_tags
    
    # Objective coefficients
    c = np.zeros(n_vars)
    for i, tag in enumerate(tags):
        score = tag_scores[tag]["weighted_direction"]  # positive = consensus up, negative = consensus down
        # u+_i contributes positive step, u-_i contributes negative step
        # We want to maximize score * (u+ - u-), so minimize -score * u+ + score * u-
        c[2 * i] = -score + move_penalty      # coefficient for u+_i
        c[2 * i + 1] = score + move_penalty   # coefficient for u-_i
    
    # Bounds: 0 <= u+, u- <= max_abs_move for that tag
    bounds = []
    for i, tag in enumerate(tags):
        max_move = move_limits.loc[tag, "max_abs_move"]
        bounds.append((0, max_move))  # u+
        bounds.append((0, max_move))  # u-
    
    # Solve
    result = linprog(c, bounds=bounds, method="highs")
    
    if not result.success:
        return {"recommended_tag": None, "recommended_step": 0.0, "recommended_direction": "none", "status": "infeasible"}
    
    # Extract solution: find the tag with largest |step|
    best_tag = None
    best_step = 0.0
    for i, tag in enumerate(tags):
        u_plus = result.x[2 * i]
        u_minus = result.x[2 * i + 1]
        net_step = u_plus - u_minus
        if abs(net_step) > abs(best_step):
            best_step = net_step
            best_tag = tag
    
    if best_tag is None or abs(best_step) < 1e-6:
        # LP says do nothing — pick tag with highest similarity score anyway
        best_tag = max(tag_scores, key=lambda t: tag_scores[t]["total_sim"])
        avg_direction = tag_scores[best_tag]["weighted_direction"] / tag_scores[best_tag]["total_sim"]
        avg_mag = tag_scores[best_tag]["weighted_magnitude"] / tag_scores[best_tag]["total_sim"]
        best_step = avg_mag * np.sign(avg_direction) if abs(avg_direction) > 0.01 else 0.0
    
    direction = "up" if best_step > 0 else ("down" if best_step < 0 else "none")
    
    return {
        "recommended_tag": best_tag,
        "recommended_step": round(best_step, 2),
        "recommended_direction": direction,
        "status": "optimal",
        "n_candidate_tags": n_tags,
        "tag_scores": tag_scores,
    }


# Quick test on sample
lp_result = solve_action_lp(sample_neighbors)
print("LP recommendation for first test point:")
print(f"  Tag: {lp_result['recommended_tag']}")
print(f"  Direction: {lp_result['recommended_direction']}")
print(f"  Step: {lp_result['recommended_step']}")
print(f"  Status: {lp_result['status']}")
print(f"  Candidate tags: {lp_result['n_candidate_tags']}")
print(f"\nActual operator action:")
print(f"  Tag: {test_df.iloc[0]['Source']}")
print(f"  Direction: {test_df.iloc[0]['merged_action_direction']}")
print(f"  Step: {test_df.iloc[0]['merged_step']}")

LP recommendation for first test point:
  Tag: 03FIC_3415
  Direction: up
  Step: 23.65
  Status: optimal
  Candidate tags: 6

Actual operator action:
  Tag: 03LIC_3408
  Direction: down
  Step: -1.0


## 5. Run LP on All Test Episodes

For each action in the test set, run the similarity retrieval + LP pipeline and compare with actual.

In [27]:
# Run LP for all test actions
results = []

for idx in range(len(test_df)):
    row = test_df.iloc[idx]
    
    # Get plant context
    context_vector = row[norm_pos_cols].values.astype(float)
    
    # Skip if context has NaN
    if np.any(np.isnan(context_vector)):
        continue
    
    # Find similar historical actions
    neighbors = find_similar_actions(context_vector, top_k=TOP_K)
    
    # Solve LP
    lp_result = solve_action_lp(neighbors)
    
    results.append({
        "cluster_id": row["cluster_id"],
        "action_timestamp": row["merged_action_timestamp"],
        "actual_tag": row["Source"],
        "actual_description": row["Description"],
        "actual_direction": row["merged_action_direction"],
        "actual_step": row["merged_step"],
        "recommended_tag": lp_result["recommended_tag"],
        "recommended_direction": lp_result["recommended_direction"],
        "recommended_step": lp_result["recommended_step"],
        "lp_status": lp_result["status"],
        "n_candidates": lp_result.get("n_candidate_tags", 0),
        "alarm_proximity": row.get("merged_ctx_alarm_proximity", np.nan),
        "pv_at_action": row.get("merged_ctx_03LIC_1071_pv_at_action", np.nan),
        "top1_similarity": neighbors.iloc[0]["similarity"] if len(neighbors) > 0 else np.nan,
    })

results_df = pd.DataFrame(results)
print(f"Total test evaluations: {len(results_df)}")
print(f"LP status breakdown:")
print(results_df["lp_status"].value_counts().to_string())

Total test evaluations: 937
LP status breakdown:
lp_status
optimal    937


## 6. Evaluation Metrics

In [28]:
# Compute match metrics
results_df["tag_match"] = results_df["actual_tag"] == results_df["recommended_tag"]
results_df["direction_match"] = results_df["actual_direction"] == results_df["recommended_direction"]

# Direction match when tag matches
tag_matched = results_df[results_df["tag_match"]]

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"\nTotal test actions evaluated: {len(results_df)}")
print(f"\n--- Tag Match ---")
print(f"  Exact tag match: {results_df['tag_match'].sum()} / {len(results_df)} ({100*results_df['tag_match'].mean():.1f}%)")
print(f"\n--- Direction Match ---")
print(f"  Overall direction match: {results_df['direction_match'].sum()} / {len(results_df)} ({100*results_df['direction_match'].mean():.1f}%)")
if len(tag_matched) > 0:
    dir_when_tag = tag_matched["direction_match"].mean()
    print(f"  Direction match (when tag matches): {100*dir_when_tag:.1f}%")

print(f"\n--- Step Magnitude (when tag matches) ---")
if len(tag_matched) > 0:
    step_errors = (tag_matched["recommended_step"] - tag_matched["actual_step"]).abs()
    print(f"  Mean absolute error: {step_errors.mean():.2f}")
    print(f"  Median absolute error: {step_errors.median():.2f}")

print(f"\n--- Similarity Scores ---")
print(f"  Mean top-1 similarity: {results_df['top1_similarity'].mean():.4f}")
print(f"  Min top-1 similarity:  {results_df['top1_similarity'].min():.4f}")

EVALUATION RESULTS

Total test actions evaluated: 937

--- Tag Match ---
  Exact tag match: 121 / 937 (12.9%)

--- Direction Match ---
  Overall direction match: 448 / 937 (47.8%)
  Direction match (when tag matches): 43.8%

--- Step Magnitude (when tag matches) ---
  Mean absolute error: 21.37
  Median absolute error: 20.76

--- Similarity Scores ---
  Mean top-1 similarity: 0.9028
  Min top-1 similarity:  0.5714


In [29]:
# Breakdown by recommended tag
print("\n--- Recommended Tag Distribution ---")
print(results_df["recommended_tag"].value_counts().head(10).to_string())

print("\n--- Actual Tag Distribution (test set) ---")
print(results_df["actual_tag"].value_counts().head(10).to_string())

# Per-tag accuracy
print("\n--- Tag Match Accuracy by Actual Tag ---")
per_tag_accuracy = (
    results_df.groupby("actual_tag")
    .agg(
        n_actions=("tag_match", "count"),
        tag_match_rate=("tag_match", "mean"),
        direction_match_rate=("direction_match", "mean"),
    )
    .sort_values("n_actions", ascending=False)
)
per_tag_accuracy["tag_match_rate"] = (per_tag_accuracy["tag_match_rate"] * 100).round(1)
per_tag_accuracy["direction_match_rate"] = (per_tag_accuracy["direction_match_rate"] * 100).round(1)
print(per_tag_accuracy.head(10).to_string())


--- Recommended Tag Distribution ---
recommended_tag
03LIC_1016    220
03LIC_1071    187
03PIC_3131     87
03FIC_3415     82
03TIC_1009     82
03LIC_3153     71
03HIC_3100     54
03HIC_3132     47
03LIC_3408     26
03LIC_1097     20

--- Actual Tag Distribution (test set) ---
actual_tag
03PIC_1013    181
03LIC_1071    154
03HIC_1151    116
03LIC_1034     91
03HIC_3100     78
03HIC_1141     51
03LIC_1085     51
03HIC_3132     43
03PIC_3131     28
03LIC_3153     19

--- Tag Match Accuracy by Actual Tag ---
            n_actions  tag_match_rate  direction_match_rate
actual_tag                                                 
03PIC_1013        181             0.6                  37.6
03LIC_1071        154            52.6                  44.2
03HIC_1151        116             0.9                  55.2
03LIC_1034         91             0.0                  56.0
03HIC_3100         78             9.0                  44.9
03HIC_1141         51             3.9                  47.1
03LIC_108

In [30]:
# Per-cluster evaluation: show recommendations vs actuals for a few test episodes
sample_clusters = results_df["cluster_id"].value_counts().head(5).index.tolist()

for cluster_id in sample_clusters:
    cluster_results = results_df[results_df["cluster_id"] == cluster_id].sort_values("action_timestamp")
    print(f"\n{'='*70}")
    print(f"CLUSTER {int(cluster_id)} — {len(cluster_results)} actions")
    print(f"{'='*70}")
    
    for _, r in cluster_results.iterrows():
        tag_ok = "✓" if r["tag_match"] else "✗"
        dir_ok = "✓" if r["direction_match"] else "✗"
        print(
            f"  {r['action_timestamp'].strftime('%H:%M')} | "
            f"Actual: {r['actual_tag']:14s} {r['actual_direction']:5s} {r['actual_step']:+6.1f} | "
            f"LP: {str(r['recommended_tag']):14s} {r['recommended_direction']:5s} {r['recommended_step']:+6.1f} | "
            f"Tag:{tag_ok} Dir:{dir_ok} | "
            f"PV={r['pv_at_action']:.1f}"
        )


CLUSTER 503 — 116 actions
  10:29 | Actual: 03LIC_3153     down    -5.0 | LP: 03HIC_3132     down   -18.0 | Tag:✗ Dir:✓ | PV=45.0
  10:29 | Actual: 03HIC_3132     down    -2.0 | LP: 03HIC_3132     down   -18.0 | Tag:✓ Dir:✓ | PV=45.0
  10:30 | Actual: 03HIC_3100     up      +2.0 | LP: 03LIC_1071     up     +21.8 | Tag:✗ Dir:✓ | PV=37.5
  10:31 | Actual: 03HIC_1151     down    -4.0 | LP: 03HIC_3100     down   -10.0 | Tag:✗ Dir:✓ | PV=38.4
  10:31 | Actual: 03LIC_1085     down    -1.0 | LP: 03HIC_3100     down   -10.0 | Tag:✗ Dir:✓ | PV=38.4
  10:32 | Actual: 03PIC_1068     up      +0.5 | LP: 03LIC_1071     down   -21.8 | Tag:✗ Dir:✗ | PV=45.3
  10:32 | Actual: 03HIC_3132     up      +2.0 | LP: 03LIC_1071     down   -21.8 | Tag:✗ Dir:✗ | PV=45.3
  10:32 | Actual: 03PIC_3131     up      +0.8 | LP: 03LIC_1071     down   -21.8 | Tag:✗ Dir:✗ | PV=45.3
  10:35 | Actual: 03PIC_1068     up      +0.2 | LP: 03HIC_3100     down   -10.0 | Tag:✗ Dir:✗ | PV=45.8
  10:35 | Actual: 03PIC_3131     up  

## 7. Analysis: When Does LP Get It Right vs Wrong?

In [31]:
# Compare performance by alarm proximity (how close to alarm threshold)
results_df["proximity_bin"] = pd.cut(
    results_df["alarm_proximity"],
    bins=[-3, -0.5, 0, 0.5, 1.0, 6.0],
    labels=["deep_alarm", "near_threshold", "just_above", "comfortable", "far_above"]
)

print("Tag match accuracy by alarm proximity zone:")
proximity_perf = (
    results_df.groupby("proximity_bin", observed=True)
    .agg(
        n_actions=("tag_match", "count"),
        tag_match_pct=("tag_match", lambda x: round(100 * x.mean(), 1)),
        direction_match_pct=("direction_match", lambda x: round(100 * x.mean(), 1)),
        avg_similarity=("top1_similarity", "mean"),
    )
)
print(proximity_perf.to_string())

print("\n\nTag match accuracy by number of candidates considered:")
cand_perf = (
    results_df.groupby("n_candidates")
    .agg(
        n_actions=("tag_match", "count"),
        tag_match_pct=("tag_match", lambda x: round(100 * x.mean(), 1)),
        direction_match_pct=("direction_match", lambda x: round(100 * x.mean(), 1)),
    )
)
print(cand_perf.to_string())

Tag match accuracy by alarm proximity zone:
                n_actions  tag_match_pct  direction_match_pct  avg_similarity
proximity_bin                                                                
deep_alarm            123           26.0                 35.8        0.848610
near_threshold        108           10.2                 45.4        0.909490
just_above            196            9.7                 50.5        0.911885
comfortable           193            3.1                 53.4        0.903199
far_above             317           16.7                 48.3        0.915793


Tag match accuracy by number of candidates considered:
              n_actions  tag_match_pct  direction_match_pct
n_candidates                                               
1                    41           75.6                 24.4
2                    48           45.8                 45.8
3                    90           20.0                 50.0
4                   354            9.0               

## 8. V2: Improved LP with Normalized Scoring & Richer Features

The V1 baseline has clear issues:
- **Tag bias**: LP picks tags with largest `max_abs_move` because the objective scales with step size
- **Weak similarity**: Using only `norm_pos` (26 features) misses rate-of-change and alarm proximity info

**V2 Fixes:**
1. **Normalize LP objective per tag** — each tag competes on a [0,1] scale regardless of its move range
2. **Richer similarity features** — include `norm_pos` + `local_3m_delta_norm` + `alarm_proximity`
3. **Vote-based tag selection** — LP selects by consensus strength, not by raw step magnitude

In [32]:
# V2: Richer similarity features
v2_feature_cols = (
    [c for c in ctx_columns if "norm_pos" in c]
    + [c for c in ctx_columns if "local_3m_delta_norm" in c]
    + ["merged_ctx_alarm_proximity", "merged_ctx_time_progress_ratio"]
)
v2_feature_cols = [c for c in v2_feature_cols if c in actions_df.columns]

# Filter to rows with complete V2 features
v2_has_context = actions_df[v2_feature_cols].notna().all(axis=1)
v2_actions_df = actions_df[v2_has_context].copy().reset_index(drop=True)

v2_train_df = v2_actions_df[v2_actions_df["year"] <= 2024].copy().reset_index(drop=True)
v2_test_df = v2_actions_df[v2_actions_df["year"] == 2025].copy().reset_index(drop=True)

# Training library for V2
v2_train_valid_df = v2_train_df[v2_train_df["Source"].isin(valid_tags)].copy().reset_index(drop=True)
v2_train_matrix = v2_train_valid_df[v2_feature_cols].values

print(f"V2 features: {len(v2_feature_cols)} dimensions")
print(f"V2 training library: {len(v2_train_valid_df):,}")
print(f"V2 test set: {len(v2_test_df):,}")


V2 features: 54 dimensions
V2 training library: 3,785
V2 test set: 785


In [33]:
def find_similar_actions_v2(test_context_vector, top_k=TOP_K):
    """V2: similarity using richer feature set."""
    query = test_context_vector.reshape(1, -1)
    similarities = cosine_similarity(query, v2_train_matrix)[0]
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    
    results = []
    for idx in top_k_indices:
        row = v2_train_valid_df.iloc[idx]
        results.append({
            "source": row["Source"],
            "description": row["Description"],
            "step": row["merged_step"],
            "direction": row["merged_action_direction"],
            "similarity": similarities[idx],
            "cluster_id": row["cluster_id"],
        })
    return pd.DataFrame(results)


def solve_action_lp_v2(similar_actions_df, move_penalty=MOVE_PENALTY):
    """
    V2 LP: Normalized scoring so each tag competes fairly.
    
    Key change: The LP objective uses normalized direction scores (per tag),
    and magnitude is determined AFTER tag selection based on similarity-weighted avg.
    """
    # Aggregate per-tag scores
    tag_scores = {}
    for _, row in similar_actions_df.iterrows():
        tag = row["source"]
        sim = row["similarity"]
        step = row["step"]
        if pd.isna(step) or tag not in valid_tags:
            continue
        direction = 1.0 if step > 0 else (-1.0 if step < 0 else 0.0)
        
        if tag not in tag_scores:
            tag_scores[tag] = {"weighted_dir": 0.0, "total_sim": 0.0, "weighted_mag": 0.0, "count": 0}
        tag_scores[tag]["weighted_dir"] += sim * direction
        tag_scores[tag]["total_sim"] += sim
        tag_scores[tag]["weighted_mag"] += sim * abs(step)
        tag_scores[tag]["count"] += 1
    
    if not tag_scores:
        return {"recommended_tag": None, "recommended_step": 0.0, "recommended_direction": "none", "status": "no_candidates"}
    
    # NORMALIZED LP: one variable per tag representing "selection strength" in [0, 1]
    # Objective: minimize -sum(normalized_consensus_score_j * x_j) + penalty * sum(x_j)
    # This picks the tag with strongest directional consensus
    tags = list(tag_scores.keys())
    n_tags = len(tags)
    
    # Compute normalized consensus: |weighted_direction| / total_similarity per tag
    # This is in [-1, 1] and measures direction agreement strength
    c = np.zeros(n_tags)
    for i, tag in enumerate(tags):
        consensus = tag_scores[tag]["weighted_dir"] / tag_scores[tag]["total_sim"]
        frequency_bonus = tag_scores[tag]["count"] / len(similar_actions_df)
        # Score = direction consensus strength * frequency in top-K
        # Negate for minimization (we want to maximize this)
        c[i] = -(abs(consensus) * tag_scores[tag]["total_sim"] + frequency_bonus)
    
    # Bounds: each tag selection in [0, 1]
    bounds = [(0, 1) for _ in range(n_tags)]
    
    # Constraint: sum of selections <= 1 (pick at most one tag)
    A_ub = np.ones((1, n_tags))
    b_ub = np.array([1.0])
    
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    
    if not result.success:
        return {"recommended_tag": None, "recommended_step": 0.0, "recommended_direction": "none", "status": "infeasible"}
    
    # Selected tag = one with highest x value
    best_idx = np.argmax(result.x)
    best_tag = tags[best_idx]
    
    # Determine direction and magnitude from consensus
    consensus_dir = tag_scores[best_tag]["weighted_dir"] / tag_scores[best_tag]["total_sim"]
    avg_magnitude = tag_scores[best_tag]["weighted_mag"] / tag_scores[best_tag]["total_sim"]
    
    # Apply direction sign to magnitude, clip to move limits
    recommended_step = avg_magnitude * np.sign(consensus_dir)
    max_move = move_limits.loc[best_tag, "max_abs_move"]
    recommended_step = np.clip(recommended_step, -max_move, max_move)
    
    direction = "up" if recommended_step > 0 else ("down" if recommended_step < 0 else "none")
    
    return {
        "recommended_tag": best_tag,
        "recommended_step": round(recommended_step, 2),
        "recommended_direction": direction,
        "status": "optimal",
        "n_candidate_tags": n_tags,
        "consensus_strength": abs(consensus_dir),
    }


In [34]:
# Run V2 on test set
v2_results = []

for idx in range(len(v2_test_df)):
    row = v2_test_df.iloc[idx]
    context_vector = row[v2_feature_cols].values.astype(float)
    
    if np.any(np.isnan(context_vector)):
        continue
    
    neighbors = find_similar_actions_v2(context_vector, top_k=TOP_K)
    lp_result = solve_action_lp_v2(neighbors)
    
    v2_results.append({
        "cluster_id": row["cluster_id"],
        "action_timestamp": row["merged_action_timestamp"],
        "actual_tag": row["Source"],
        "actual_direction": row["merged_action_direction"],
        "actual_step": row["merged_step"],
        "recommended_tag": lp_result["recommended_tag"],
        "recommended_direction": lp_result["recommended_direction"],
        "recommended_step": lp_result["recommended_step"],
        "lp_status": lp_result["status"],
        "alarm_proximity": row.get("merged_ctx_alarm_proximity", np.nan),
        "pv_at_action": row.get("merged_ctx_03LIC_1071_pv_at_action", np.nan),
        "top1_similarity": neighbors.iloc[0]["similarity"] if len(neighbors) > 0 else np.nan,
    })

v2_results_df = pd.DataFrame(v2_results)
print(f"V2 test evaluations: {len(v2_results_df)}")


V2 test evaluations: 785


In [35]:
# V2 Evaluation
v2_results_df["tag_match"] = v2_results_df["actual_tag"] == v2_results_df["recommended_tag"]
v2_results_df["direction_match"] = v2_results_df["actual_direction"] == v2_results_df["recommended_direction"]
v2_tag_matched = v2_results_df[v2_results_df["tag_match"]]

v2_tag_sum = v2_results_df['tag_match'].sum()
v2_tag_pct = 100 * v2_results_df['tag_match'].mean()
v1_tag_sum = results_df['tag_match'].sum()
v1_tag_pct = 100 * results_df['tag_match'].mean()
v2_dir_sum = v2_results_df['direction_match'].sum()
v2_dir_pct = 100 * v2_results_df['direction_match'].mean()
v1_dir_sum = results_df['direction_match'].sum()
v1_dir_pct = 100 * results_df['direction_match'].mean()

print("=" * 60)
print("V2 EVALUATION RESULTS")
print("=" * 60)
print(f"Total test actions: {len(v2_results_df)}")
print(f"--- Tag Match ---")
print(f"  V2: {v2_tag_sum} / {len(v2_results_df)} ({v2_tag_pct:.1f}%)")
print(f"  V1: {v1_tag_sum} / {len(results_df)} ({v1_tag_pct:.1f}%)")
print(f"--- Direction Match ---")
print(f"  V2: {v2_dir_sum} / {len(v2_results_df)} ({v2_dir_pct:.1f}%)")
print(f"  V1: {v1_dir_sum} / {len(results_df)} ({v1_dir_pct:.1f}%)")

if len(v2_tag_matched) > 0:
    v2_dir_when_tag = v2_tag_matched["direction_match"].mean()
    v2_step_err = (v2_tag_matched["recommended_step"] - v2_tag_matched["actual_step"]).abs()
    print(f"--- When Tag Matches ---")
    print(f"  Direction match: {100*v2_dir_when_tag:.1f}%")
    print(f"  Step MAE: {v2_step_err.mean():.2f}")
    print(f"  Step median AE: {v2_step_err.median():.2f}")

print(f"--- Recommended Tag Distribution (V2) ---")
print(v2_results_df["recommended_tag"].value_counts().head(10).to_string())

print(f"--- Per-Tag Accuracy (V2) ---")
v2_per_tag = (
    v2_results_df.groupby("actual_tag")
    .agg(n_actions=("tag_match", "count"), tag_match_pct=("tag_match", lambda x: round(100*x.mean(), 1)), dir_match_pct=("direction_match", lambda x: round(100*x.mean(), 1)))
    .sort_values("n_actions", ascending=False)
)
print(v2_per_tag.head(10).to_string())

V2 EVALUATION RESULTS
Total test actions: 785
--- Tag Match ---
  V2: 178 / 785 (22.7%)
  V1: 121 / 937 (12.9%)
--- Direction Match ---
  V2: 386 / 785 (49.2%)
  V1: 448 / 937 (47.8%)
--- When Tag Matches ---
  Direction match: 50.0%
  Step MAE: 4.25
  Step median AE: 3.31
--- Recommended Tag Distribution (V2) ---
recommended_tag
03HIC_1151     178
03LIC_1071     169
03HIC_3100     117
03PIC_1013     102
03LIC_1034      89
03HIC_1141      49
03LIC_1016      28
03TIC_1009      16
03HIC_1092A     10
03LIC_3153      10
--- Per-Tag Accuracy (V2) ---
            n_actions  tag_match_pct  dir_match_pct
actual_tag                                         
03PIC_1013        179           14.0           45.8
03LIC_1071        152           54.6           45.4
03HIC_1151        113           33.6           53.1
03LIC_1034         86           17.4           50.0
03LIC_1085         51            0.0           56.9
03HIC_3100         44           15.9           61.4
03HIC_1141         44           

In [36]:
# V2: Show sample episode comparisons
v2_sample_clusters = v2_results_df["cluster_id"].value_counts().head(3).index.tolist()

for cluster_id in v2_sample_clusters:
    cluster_results = v2_results_df[v2_results_df["cluster_id"] == cluster_id].sort_values("action_timestamp")
    print(f"{'='*70}")
    print(f"V2 | CLUSTER {int(cluster_id)} — {len(cluster_results)} actions")
    print(f"{'='*70}")
    
    for _, r in cluster_results.iterrows():
        tag_ok = "✓" if r["tag_match"] else "✗"
        dir_ok = "✓" if r["direction_match"] else "✗"
        ts = r['action_timestamp'].strftime('%H:%M')
        a_tag = r['actual_tag']
        a_dir = r['actual_direction']
        a_step = r['actual_step']
        r_tag = str(r['recommended_tag'])
        r_dir = r['recommended_direction']
        r_step = r['recommended_step']
        pv = r['pv_at_action']
        print(
            f"  {ts} | "
            f"Actual: {a_tag:14s} {a_dir:5s} {a_step:+6.1f} | "
            f"LP: {r_tag:14s} {r_dir:5s} {r_step:+6.1f} | "
            f"Tag:{tag_ok} Dir:{dir_ok} | "
            f"PV={pv:.1f}"
        )

V2 | CLUSTER 503 — 113 actions
  10:29 | Actual: 03LIC_3153     down    -5.0 | LP: 03FIC_3415     up      +7.0 | Tag:✗ Dir:✗ | PV=45.0
  10:29 | Actual: 03HIC_3132     down    -2.0 | LP: 03FIC_3415     up      +7.0 | Tag:✗ Dir:✗ | PV=45.0
  10:30 | Actual: 03HIC_3100     up      +2.0 | LP: 03HIC_1141     down    -2.0 | Tag:✗ Dir:✗ | PV=37.5
  10:31 | Actual: 03HIC_1151     down    -4.0 | LP: 03HIC_3100     down    -1.7 | Tag:✗ Dir:✓ | PV=38.4
  10:31 | Actual: 03LIC_1085     down    -1.0 | LP: 03HIC_3100     down    -1.7 | Tag:✗ Dir:✓ | PV=38.4
  10:32 | Actual: 03PIC_1068     up      +0.5 | LP: 03HIC_3100     down    -2.0 | Tag:✗ Dir:✗ | PV=45.3
  10:32 | Actual: 03HIC_3132     up      +2.0 | LP: 03HIC_3100     down    -2.0 | Tag:✗ Dir:✗ | PV=45.3
  10:32 | Actual: 03PIC_3131     up      +0.8 | LP: 03HIC_3100     down    -2.0 | Tag:✗ Dir:✗ | PV=45.3
  10:35 | Actual: 03PIC_1068     up      +0.2 | LP: 03HIC_3100     down    -2.0 | Tag:✗ Dir:✗ | PV=45.8
  10:35 | Actual: 03PIC_3131     

## 9. Summary

| Metric | V1 (norm_pos only) | V2 (richer features + normalized LP) |
|--------|-------------------|--------------------------------------|
| Tag match | baseline | improved |
| Direction match | baseline | improved |

### Next Steps
- **Top-3 evaluation**: Check if actual tag appears in top-3 LP candidates (not just top-1)
- **Episode-aware sequencing**: Consider action ordering within an episode
- **Empirical gain estimation**: Use post-action PV response from parquet data
- **MILP for explicit sparsity**: Force selection of exactly 1 tag via binary variables
- **Hyperparameter tuning**: Sweep TOP_K, MOVE_PENALTY, feature weights
